# "Check me in 30 seconds" — the server rule on PhysioNet/CinC 2017 (Notebook 7)
**Bektur Asilov · COMSE-23 · Ala-Too**

In the app a person can press **"Check me — 30 s"**: the phone records 30 s of ECG from the sensor and the server
answers *regular rhythm*, *irregular rhythm — show this to a doctor* or *signal too poor — repeat*.
The server code (`backend/app/ecgtools.py`) is pasted below **unchanged**. It was never tuned on CinC 2017.

Rules fixed **before** the run:
* each CinC 2017 recording is cut to its **first 30 s** (shorter ones are used whole), band-pass 0.5–40 Hz and
  resampled to 125 Hz as on the device;
* the verdict is the server function `check()` itself:
  **no verdict** if fewer than 16 R-peaks are found or the signal quality index `sqi < 0.4`;
* **irregular** if `irregularity = mean|ΔRR| / mean RR > 0.10`, otherwise **regular**;
* primary result: AF (A) vs Normal (N) — AF recordings called irregular (sensitivity), Normal called regular
  (specificity), counting *no verdict* separately and also as "not found";
* reported separately: Other rhythm (O) and Noisy (~), because real users produce those too;
* comparison with the 16-beat neural model from Notebook 5 (86.3 % of AF found, 92.5 % of Normal not called AF);
* ROC-AUC of `irregularity` and the best threshold are shown for information only — they are **not** used to change the rule.

> Kaggle: Internet **On** (CinC 2017 ≈ 95 MB from the PhysioNet open-data mirror). CPU is enough.

In [ ]:
import os, time, json, subprocess, sys, zipfile
import numpy as np, pandas as pd
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
from sklearn.metrics import roc_auc_score, roc_curve
FS = 125
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
CINC_DIR = os.environ.get('CINC_DIR')
if not CINC_DIR:
    MIRRORS = ['https://physionet-open.s3.amazonaws.com/challenge-2017/1.0.0/',
               'https://physionet.org/files/challenge-2017/1.0.0/']
    os.makedirs('/kaggle/tmp', exist_ok=True)
    for f in ('REFERENCE-v3.csv', 'training2017.zip'):
        for base in MIRRORS:
            if subprocess.run(['curl', '-fsSL', '--max-time', '1200', '-o', f'/kaggle/tmp/{f}', base + f]).returncode == 0:
                print(f, 'from', base, flush=True); break
        else:
            raise RuntimeError('download failed: ' + f)
    zipfile.ZipFile('/kaggle/tmp/training2017.zip').extractall('/kaggle/tmp')
    CINC_DIR, REF = '/kaggle/tmp/training2017', '/kaggle/tmp/REFERENCE-v3.csv'
else:
    REF = os.path.join(CINC_DIR, 'REFERENCE-v3.csv')
ref = pd.read_csv(REF, header=None, names=['record', 'label'])
print(len(ref), 'recordings;', ref.label.value_counts().to_dict())

## Server code, unchanged (`backend/app/ecgtools.py` without the PNG drawing)

In [ ]:
"""ECG strip analysis on the server: R-peaks, RR intervals, Poincaré points, signal quality index, PNG image.

Pure Python (strips are short: 10 s x 125 Hz). The R-peak detector is the firmware algorithm
(derivative^2 -> 150 ms moving average -> adaptive threshold, 250 ms refractory), adapted to run offline
on a short strip: the threshold tracker is primed with the first 2 s instead of a 2-s warm-up.
"""

import io
import math
import statistics


def _highpass(x: list[float], fs: int) -> list[float]:
    """Remove baseline wander: subtract a 0.6-s centred moving mean."""
    w = max(3, int(0.6 * fs)) | 1
    h = w // 2
    n = len(x)
    pre = [0.0]
    for v in x:
        pre.append(pre[-1] + v)
    out = []
    for i in range(n):
        a, b = max(0, i - h), min(n, i + h + 1)
        out.append(x[i] - (pre[b] - pre[a]) / (b - a))
    return out


def r_peaks(x: list[float], fs: int = 125) -> list[int]:
    n = len(x)
    if n < fs:
        return []
    y = _highpass(x, fs)
    ma = max(3, int(0.15 * fs))
    env, buf, s, prev = [], [0.0] * ma, 0.0, y[0]
    for i, v in enumerate(y):
        e = (v - prev) ** 2
        prev = v
        s += e - buf[i % ma]
        buf[i % ma] = e
        env.append(s / ma)
    decay = 0.5 ** (1 / (2.0 * fs))
    thr, refr = 0.35, int(0.25 * fs)
    pk = max(env[: min(n, 2 * fs)]) or 1e-12
    out, last, armed = [], -10 ** 9, True
    for i in range(int(0.2 * fs), n):
        pk = max(pk * decay, env[i])
        if armed and env[i] > thr * pk and i - last > refr:
            # the envelope lags the QRS by ~MA/2: take the largest |amplitude| just before
            a, b = max(0, i - ma - 4), min(n, i + 3)
            j = max(range(a, b), key=lambda k: abs(y[k]))
            if not out or j - out[-1] > refr:
                out.append(j)
            last, armed = i, False
        elif not armed and env[i] < 0.5 * thr * pk:
            armed = True
    return out


def rr_intervals(peaks: list[int], fs: int = 125) -> list[float]:
    return [round((b - a) / fs, 3) for a, b in zip(peaks, peaks[1:])]


def poincare(rr: list[float]) -> dict:
    """Points (RRn, RRn+1) and the SD1/SD2 descriptors (SD1 = short-term, beat-to-beat variability)."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    return poincare_points([[a, b] for a, b in zip(rr, rr[1:])])


def poincare_points(pts: list[list[float]]) -> dict:
    if len(pts) < 3:
        return {"points": pts, "sd1": None, "sd2": None, "ratio": None}
    d1 = [(b - a) / math.sqrt(2) for a, b in pts]
    d2 = [(b + a) / math.sqrt(2) for a, b in pts]
    sd1, sd2 = statistics.pstdev(d1), statistics.pstdev(d2)
    return {"points": pts, "sd1": round(sd1 * 1000, 1), "sd2": round(sd2 * 1000, 1),
            "ratio": round(sd1 / sd2, 2) if sd2 > 0 else None}


def irregularity(rr: list[float]) -> float | None:
    """Mean |successive difference| / mean RR — high (>~0.1) in AF, low in sinus rhythm."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    if len(rr) < 4:
        return None
    return round(statistics.mean(abs(b - a) for a, b in zip(rr, rr[1:])) / statistics.mean(rr), 3)


def quality(x: list[float], fs: int = 125, peaks: list[int] | None = None) -> dict:
    """Signal quality index 0..1 from four checks a cardiologist does by eye:
    no flat line / clipping, plausible heart rate, consistent QRS shape, little high-frequency noise."""
    n = len(x)
    if n < 2 * fs:
        return {"sqi": 0.0, "level": "poor", "reasons": ["too_short"]}
    peaks = r_peaks(x, fs) if peaks is None else peaks
    y = _highpass(x, fs)
    reasons = []
    rng = max(y) - min(y)
    flat = sum(1 for a, b in zip(y, y[1:]) if abs(b - a) < 1e-4) / n
    s_flat = 0.0 if rng < 0.05 or flat > 0.5 else 1.0
    if s_flat == 0:
        reasons.append("flat")
    secs = n / fs
    bpm = len(peaks) / secs * 60
    s_rate = 1.0 if 30 <= bpm <= 200 else 0.0
    if not s_rate:
        reasons.append("rate")
    # QRS template correlation
    h = int(0.1 * fs)
    segs = [y[p - h:p + h] for p in peaks if p - h >= 0 and p + h <= n]
    if len(segs) >= 3:
        tmpl = [statistics.median(col) for col in zip(*segs)]

        def corr(a, b):
            ma, mb = statistics.mean(a), statistics.mean(b)
            num = sum((u - ma) * (v - mb) for u, v in zip(a, b))
            den = math.sqrt(sum((u - ma) ** 2 for u in a) * sum((v - mb) ** 2 for v in b)) or 1e-12
            return num / den
        s_shape = max(0.0, statistics.mean(corr(s, tmpl) for s in segs))
    else:
        s_shape = 0.0
    if s_shape < 0.7:
        reasons.append("shape")
    # high-frequency noise: energy of the second difference relative to the signal
    d2 = [y[i + 1] - 2 * y[i] + y[i - 1] for i in range(1, n - 1)]
    noise = (sum(v * v for v in d2) / len(d2)) ** 0.5 / (statistics.pstdev(y) or 1e-12)
    s_noise = max(0.0, min(1.0, 1.5 - noise))
    if s_noise < 0.5:
        reasons.append("noise")
    sqi = s_flat * s_rate * (0.6 * s_shape + 0.4 * s_noise)
    sqi = round(max(0.0, min(1.0, sqi)), 2)
    return {"sqi": sqi, "level": "good" if sqi >= 0.7 else "fair" if sqi >= 0.4 else "poor", "reasons": reasons}


PAUSE_S = 3.0   # an RR gap of 3 s or more is a pause worth showing to a doctor


def pauses(peaks: list[int], fs: int = 125) -> list[dict]:
    return [{"at_s": round(a / fs, 2), "rr_s": round((b - a) / fs, 2)}
            for a, b in zip(peaks, peaks[1:]) if (b - a) / fs >= PAUSE_S]


def analyze(x: list[float], fs: int = 125) -> dict:
    p = r_peaks(x, fs)
    rr = rr_intervals(p, fs)
    q = quality(x, fs, p)
    # a pause only counts when the rest of the strip is clean; otherwise it is more likely a lost electrode
    return {"r_peaks": p, "rr": rr, "poincare": poincare(rr), "irregularity": irregularity(rr),
            "quality": q, "pauses": pauses(p, fs) if q["sqi"] >= 0.4 else []}


# "Check me in 30 seconds" (wellness wording): rules fixed before validation, see ml/CHECK30_CINC2017.md
CHECK_MIN_BEATS, CHECK_MIN_SQI, CHECK_IRR = 16, 0.4, 0.10


def check(x: list[float], fs: int = 125) -> dict:
    """Verdict for a 30-s spot check: "regular", "irregular" (show a doctor) or "no_verdict" (repeat)."""
    a = analyze(x, fs)
    n, sqi, irr = len(a["r_peaks"]), a["quality"]["sqi"], a["irregularity"]
    if n < CHECK_MIN_BEATS or sqi < CHECK_MIN_SQI or irr is None:
        verdict = "no_verdict"
    else:
        verdict = "irregular" if irr > CHECK_IRR else "regular"
    bpm = round(len(a["r_peaks"]) / (len(x) / fs) * 60) if x else None
    return {"verdict": verdict, "beats": n, "sqi": sqi, "irregularity": irr, "bpm": bpm,
            "reasons": a["quality"]["reasons"]}

In [ ]:
def check30(x):
    c = check(list(map(float, x)), FS)          # the server function itself, constants inside it
    return c['verdict'], c['beats'], c['sqi'], c['irregularity']

print('rule:', CHECK_MIN_BEATS, 'beats,', 'sqi >=', CHECK_MIN_SQI, ', irregular if >', CHECK_IRR)

def to_125(sig, fs):
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

rows, T0 = [], time.time()
for i, (rec, lab) in enumerate(zip(ref.record, ref.label)):
    r = wfdb.rdrecord(os.path.join(CINC_DIR, rec))
    x = to_125(r.p_signal[:30 * r.fs, 0], r.fs)
    v, n, sqi, irr = check30(x)
    rows.append(dict(record=rec, label=lab, dur_s=len(x) / FS, beats=n, sqi=sqi, irregularity=irr, verdict=v))
    if (i + 1) % 1000 == 0: print(f'{i+1}/{len(ref)}  [{time.time()-T0:.0f} s]', flush=True)
R = pd.DataFrame(rows); R.to_csv(f'{OUT_DIR}/check30_predictions.csv', index=False)
print(f'done in {time.time()-T0:.0f} s')

## Results

In [ ]:
tab = pd.crosstab(R.label, R.verdict, normalize='index').round(3)
print(pd.crosstab(R.label, R.verdict)); print(tab)
A, N = R[R.label == 'A'], R[R.label == 'N']
Av, Nv = A[A.verdict != 'no_verdict'], N[N.verdict != 'no_verdict']
both = R[R.label.isin(['A', 'N']) & R.irregularity.notna()]
auc = roc_auc_score(both.label == 'A', both.irregularity)
fpr, tpr, thr = roc_curve(both.label == 'A', both.irregularity); k = int(np.argmax(tpr - fpr))
S = {
  'recordings_A': len(A), 'recordings_N': len(N),
  'no_verdict_A': float((A.verdict == 'no_verdict').mean()), 'no_verdict_N': float((N.verdict == 'no_verdict').mean()),
  'sensitivity_with_verdict': float((Av.verdict == 'irregular').mean()),
  'specificity_with_verdict': float((Nv.verdict == 'regular').mean()),
  'AF_found_of_all_AF': float((A.verdict == 'irregular').mean()),
  'Normal_not_called_irregular': float((N.verdict != 'irregular').mean()),
  'Other_called_irregular': float((R[R.label == 'O'].verdict == 'irregular').mean()),
  'Noisy_no_verdict': float((R[R.label == '~'].verdict == 'no_verdict').mean()),
  'Noisy_called_irregular': float((R[R.label == '~'].verdict == 'irregular').mean()),
  'irregularity_auc_A_vs_N': float(auc),
  'info_only_best_threshold_youden': float(thr[k]), 'info_only_sens_spec_at_best': [float(tpr[k]), float(1 - fpr[k])],
  'compare_16beat_model_nb5': {'AF_found_of_all_AF': 0.863, 'Normal_not_called_AF': 0.925},
}
json.dump(S, open(f'{OUT_DIR}/check30_summary.json', 'w'), indent=2)
print(json.dumps(S, indent=2, default=lambda v: round(float(v), 4)))